# Model_ResNet_Pretrained

ResNet50 fine-tuned from ImageNet weights on VWW using three-phase progressive unfreezing.

**Three-phase protocol (mirrors VGG_Pretrained):**
- Phase 1 (epochs 1–9): entire backbone frozen, train head only at `lr=3e-4`
- Phase 2 (epochs 10–19): unfreeze `layer3 + layer4` at `lr=1e-4`
- Phase 3 (epochs 20–30): unfreeze all backbone layers at `lr=3e-5`

**Note:** ResNet50 has ~25M parameters vs VGG16-BN's ~138M.  
Both are over-parameterized for this 7k-image binary task, but ResNet50 is  
significantly leaner and generalizes better on small datasets.  
Compare `vgg_pretrained_seed_85` vs `resnet_pretrained_seed_85` to see which makes the stronger KD teacher.

**Prerequisites:** Run `Model_MobileNetV2.ipynb` and `Model_VGG_Pretrained.ipynb` first.

In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────
import os
import numpy as np
import torch

from utils.dataset import prepare_dataset, get_loaders
from utils.models  import ResNet_Pretrained, count_params, model_size_mb
from utils.train   import setup_device, evaluate, train_multi_seed, plot_history

device = setup_device(seed=41)

In [ ]:
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")

In [ ]:
SAVE_DIR = "/content/drive/My Drive/stm32-thesis/checkpoints"

In [ ]:
# ── Parameter count ─────────────────────────────────────────────────
model_tmp = ResNet_Pretrained().to(device)
total, trainable = count_params(model_tmp)
print(f"Total params    : {total:,}")
print(f"Trainable params: {trainable:,}  (head only at start)")
print(f"Model size      : {model_size_mb(model_tmp):.1f} MB")
del model_tmp

In [ ]:
# ── Train across 5 seeds ─────────────────────────────────────────────
results, best = train_multi_seed(
    model_fn     = ResNet_Pretrained,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    seeds        = [41, 52, 63, 74, 85],
    save_dir     = SAVE_DIR,
    name_prefix  = "resnet_pretrained",
    pretrained   = True,
    # Three-phase hyperparameters
    epochs       = 30,
    lr_phase1    = 3e-4,
    lr_phase2    = 1e-4,
    lr_phase3    = 3e-5,
    phase2_epoch = 10,
    phase3_epoch = 20,
    weight_decay    = 1e-4,
    label_smoothing = 0.1,
    patience        = 10,
)

In [ ]:
plot_history(best, title=f"ResNet50 Pretrained (seed {best['seed']})")

accs = [r['best_acc'] for r in results]
print(f"\nResNet50 Pretrained  |  Mean: {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%  |  Best: {best['best_acc']*100:.2f}% (seed {best['seed']})")
print(f"Best checkpoint: {best['save_path']}")

---
## ResNet50 vs VGG16-BN Pretrained
Run `Model_VGG_Pretrained.ipynb` first, then compare teacher candidates below.

In [ ]:
# ── Teacher candidate comparison ─────────────────────────────────────
from utils.models import VGG_Pretrained

VGG_CKPT = f"{SAVE_DIR}/vgg_pretrained_seed_85.pth"   # adjust seed if needed

resnet_model = ResNet_Pretrained().to(device)
resnet_model.load_state_dict(torch.load(best['save_path'], map_location=device))
resnet_acc = evaluate(resnet_model, val_loader, device)

if os.path.exists(VGG_CKPT):
    vgg_model = VGG_Pretrained().to(device)
    vgg_model.load_state_dict(torch.load(VGG_CKPT, map_location=device))
    vgg_acc = evaluate(vgg_model, val_loader, device)
else:
    vgg_acc = None
    print("⚠️  VGG_Pretrained checkpoint not found — run Model_VGG_Pretrained first")

resnet_total, _ = count_params(resnet_model)

print("=" * 52)
print(f"{'Model':<28} {'Val Acc':>9} {'Params':>12}")
print("-" * 52)
print(f"{'ResNet50 Pretrained':<28} {resnet_acc*100:>8.2f}% {resnet_total:>12,}")
if vgg_acc is not None:
    vgg_total, _ = count_params(vgg_model)
    print(f"{'VGG16-BN Pretrained':<28} {vgg_acc*100:>8.2f}% {vgg_total:>12,}")
    print("-" * 52)
    print(f"{'Gap (ResNet - VGG)':<28} {(resnet_acc - vgg_acc)*100:>+8.2f}%")
print("=" * 52)